# N-HiTS: многомасштабная иерархия

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/12_nhits.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q neuralforecast pandas numpy matplotlib

## Подготовка данных

In [ ]:
import pandas as pd
import numpy as np

# Создаём синтетические данные для демонстрации
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')
y = 100 + np.cumsum(np.random.randn(365)) + 20 * np.sin(np.arange(365) / 7 * 2 * np.pi)

train = pd.DataFrame({
    'unique_id': 'series_1',
    'ds': dates,
    'y': y
})
print(train.head())

## N-HiTS: конфигурация и обучение

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MAE

# Параметры
HORIZON = 16
SEASON_LENGTH = 7

# Конфигурация N-HiTS
# Три стека с разными масштабами
model = NHITS(
    h=HORIZON,
    input_size=4 * HORIZON,              # более длинное окно помогает низкочастотным стекам
    loss=MAE(),
    max_steps=1000,
    
    # Конфигурация стеков
    n_pool_kernel_size=[1, 2, 4],        # kernel sizes для MaxPool
    n_freq_downsample=[1, 2, 4],         # expressiveness ratios (обратные значения)
    stack_types=['identity'] * 3,         # тип базиса (identity = generic)
    n_blocks=[1, 1, 1],                   # по одному блоку в стеке
    mlp_units=[[256, 256], [256, 256], [256, 256]],
    
    scaler_type='standard',
    random_seed=42
)

# Обучаем
nf = NeuralForecast(
    models=[model],
    freq='D'
)
nf.fit(df=train)

# Прогнозируем
forecasts = nf.predict()
print(forecasts)

## Сравнение с N-BEATS

In [ ]:
from neuralforecast.models import NBEATS

# N-BEATS с аналогичной конфигурацией
model_nbeats = NBEATS(
    h=HORIZON,
    input_size=4 * HORIZON,
    loss=MAE(),
    max_steps=1000,
    stack_types=['generic'] * 3,
    n_blocks=[1, 1, 1],
    mlp_units=[[256, 256], [256, 256], [256, 256]],
    scaler_type='standard',
    random_seed=42
)

# N-HiTS
model_nhits = NHITS(
    h=HORIZON,
    input_size=4 * HORIZON,
    loss=MAE(),
    max_steps=1000,
    n_pool_kernel_size=[1, 2, 4],
    n_freq_downsample=[1, 2, 4],
    stack_types=['identity'] * 3,
    n_blocks=[1, 1, 1],
    mlp_units=[[256, 256], [256, 256], [256, 256]],
    scaler_type='standard',
    random_seed=42
)

# Обучаем обе модели
nf_compare = NeuralForecast(
    models=[model_nhits, model_nbeats],
    freq='D'
)
nf_compare.fit(df=train)

# Прогнозируем
forecasts_compare = nf_compare.predict()

# forecasts_compare содержит колонки: 
# unique_id, ds, NHITS, NBEATS
print(forecasts_compare)

## Подбор масштабов под данные

In [ ]:
def suggest_pool_kernels(season_length, horizon, n_stacks=3):
    """
    Эвристика для выбора kernel sizes на основе 
    сезонности и горизонта.
    
    Идея: каждый следующий стек работает на масштабе,
    кратном предыдущему.
    """
    kernels = [1]  # первый стек всегда на полном разрешении
    
    current = 1
    for _ in range(n_stacks - 1):
        # Следующий масштаб — либо сезонность, 
        # либо удвоение предыдущего
        if current < season_length:
            next_kernel = min(season_length, current * 2)
        else:
            next_kernel = current * 2
        
        # Не превышаем горизонт
        if next_kernel > horizon:
            next_kernel = horizon
            
        kernels.append(next_kernel)
        current = next_kernel
    
    return kernels

# Для дневных данных с недельной сезонностью
kernels = suggest_pool_kernels(
    season_length=7, 
    horizon=16, 
    n_stacks=3
)
print(f"Suggested kernels: {kernels}")
# Output: Suggested kernels: [1, 2, 7]

## Визуализация

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))

# История
history = train.tail(50)
ax.plot(history['ds'], history['y'], label='История', color='blue')

# Прогнозы
ax.plot(forecasts_compare['ds'], forecasts_compare['NHITS'], 
        label='N-HiTS', linestyle='--', color='red')
ax.plot(forecasts_compare['ds'], forecasts_compare['NBEATS'], 
        label='N-BEATS', linestyle='--', color='green')

ax.set_title('N-HiTS vs N-BEATS')
ax.set_xlabel('Дата')
ax.set_ylabel('Значение')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()